In [1]:
%pip -q install duckdb pyarrow

from google.colab import drive
from pathlib import Path
import os
import shutil

import duckdb
import pandas as pd
import pyarrow.parquet as pq

try:
    drive.flush_and_unmount()
except Exception:
    pass

if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

drive.mount(
    "/content/drive",
    force_remount=True,
    timeout_ms=300000,
)

DATA_DIR = Path("/content/drive/MyDrive/Language Detection")
PARQUET_PATH = DATA_DIR / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"

if not DATA_DIR.exists():
    raise FileNotFoundError(DATA_DIR)

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

parquet_file = pq.ParquetFile(PARQUET_PATH)
metadata = parquet_file.metadata

print("File path  :", PARQUET_PATH)
print("File size  :", f"{PARQUET_PATH.stat().st_size / (1024**2):.2f} MB")
print("Rows       :", f"{metadata.num_rows:,}")
print("Row groups :", metadata.num_row_groups)
print("Columns    :", metadata.num_columns)


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
File path  : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
File size  : 469.49 MB
Rows       : 3,469
Row groups : 1
Columns    : 12


In [2]:
LANGUAGE_CODES = ("en", "de", "fr", "pt", "es", "ru")
SESSIONS_PER_LANGUAGE = 5
SEGMENTS_PER_SESSION = 5
SEGMENTS_PER_LANGUAGE = SESSIONS_PER_LANGUAGE * SEGMENTS_PER_SESSION
MIN_WORDS = 4
MAX_WORDS = 12
MIN_SESSION_SEGMENTS = 25
MIDDLE_START = 0.25
MIDDLE_END = 0.75

DB_PATH = Path("/content/language_review.duckdb")
TEMP_DIR = Path("/content/duckdb_tmp")

TEMP_DIR.mkdir(parents=True, exist_ok=True)

if DB_PATH.exists():
    DB_PATH.unlink()

con = duckdb.connect(DB_PATH.as_posix())
con.execute("SET threads TO 4")
con.execute("SET memory_limit = '4GB'")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order = false")

language_sql = ", ".join(f"'{code}'" for code in LANGUAGE_CODES)

con.execute(
    f"""
    CREATE TABLE segment_pool AS
    WITH source AS (
        SELECT
            gamesession_id,
            url,
            TRY_CAST(created_at AS TIMESTAMP) AS created_at,
            lang_detected,
            len(transcript_segments) AS session_segment_count,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected IN ({language_sql})
            AND transcript_segments IS NOT NULL
            AND len(transcript_segments) >= {MIN_SESSION_SEGMENTS}
            AND lower(url) LIKE '%youtube%'
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            created_at,
            lang_detected,
            session_segment_count,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM source
    ),
    parsed AS (
        SELECT
            gamesession_id,
            url,
            created_at,
            lang_detected,
            session_segment_count,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            regexp_extract_all(
                TRIM(segment.text),
                '[\\p{{L}}\\p{{N}}][\\p{{L}}\\p{{M}}\\p{{N}}''’_-]*[\\p{{L}}\\p{{M}}\\p{{N}}]'
            ) AS valid_words
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
    ),
    filtered AS (
        SELECT
            gamesession_id,
            url,
            created_at,
            lang_detected,
            session_segment_count,
            segment_index,
            segment_start,
            segment_end,
            segment_text,
            len(valid_words) AS word_count,
            TRIM(
                regexp_replace(
                    lower(
                        regexp_replace(
                            segment_text,
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}''’_-]+',
                            ' ',
                            'g'
                        )
                    ),
                    '\\s+',
                    ' ',
                    'g'
                )
            ) AS normalized_text,
            abs(
                segment_index
                - ((session_segment_count + 1) / 2.0)
            ) AS center_distance
        FROM parsed
        WHERE
            len(valid_words) BETWEEN {MIN_WORDS} AND {MAX_WORDS}
    ),
    bounded AS (
        SELECT *
        FROM filtered
        WHERE
            segment_start IS NOT NULL
            AND segment_end IS NOT NULL
            AND normalized_text <> ''
            AND segment_index >= ceil(session_segment_count * {MIDDLE_START})
            AND segment_index <= floor(session_segment_count * {MIDDLE_END})
    ),
    deduplicated AS (
        SELECT *
        FROM bounded
        QUALIFY row_number() OVER (
            PARTITION BY
                lang_detected,
                gamesession_id,
                normalized_text
            ORDER BY
                center_distance ASC,
                segment_index ASC
        ) = 1
    )
    SELECT *
    FROM deduplicated
    """
)

con.execute(
    """
    CREATE INDEX segment_pool_language_session_idx
    ON segment_pool(lang_detected, gamesession_id)
    """
)


def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def get_language_segments(language_code):
    if language_code not in LANGUAGE_CODES:
        raise ValueError(language_code)

    candidates = con.execute(
        """
        SELECT
            gamesession_id,
            url,
            created_at,
            segment_index,
            segment_start,
            segment_end,
            segment_text,
            normalized_text,
            word_count,
            lang_detected AS language_detected,
            center_distance
        FROM segment_pool
        WHERE lang_detected = ?
        ORDER BY
            created_at DESC,
            gamesession_id DESC,
            center_distance ASC,
            segment_index ASC
        """,
        [language_code],
    ).df()

    if candidates.empty:
        raise ValueError(f"No qualifying YouTube segments for {language_code}")

    session_order = (
        candidates[
            ["gamesession_id", "created_at"]
        ]
        .drop_duplicates("gamesession_id")
        .sort_values(
            ["created_at", "gamesession_id"],
            ascending=[False, False],
            na_position="last",
        )
        ["gamesession_id"]
        .tolist()
    )

    selected = []
    seen_transcripts = set()

    for gamesession_id in session_order:
        session_candidates = candidates[
            candidates["gamesession_id"] == gamesession_id
        ]

        session_candidates = session_candidates[
            ~session_candidates["normalized_text"].isin(seen_transcripts)
        ]

        if len(session_candidates) < SEGMENTS_PER_SESSION:
            continue

        picked = (
            session_candidates
            .head(SEGMENTS_PER_SESSION)
            .sort_values("segment_index")
            .reset_index(drop=True)
        )

        selected.append(picked)
        seen_transcripts.update(
            picked["normalized_text"].tolist()
        )

        if len(selected) == SESSIONS_PER_LANGUAGE:
            break

    if len(selected) != SESSIONS_PER_LANGUAGE:
        raise ValueError(
            f"Not enough qualifying YouTube sessions for {language_code}: "
            f"{len(selected)}/{SESSIONS_PER_LANGUAGE}"
        )

    result = pd.concat(
        selected,
        ignore_index=True,
    )

    if len(result) != SEGMENTS_PER_LANGUAGE:
        raise ValueError(
            f"Expected {SEGMENTS_PER_LANGUAGE} segments for "
            f"{language_code}, found {len(result)}"
        )

    session_counts = (
        result
        .groupby("gamesession_id")
        .size()
    )

    if len(session_counts) != SESSIONS_PER_LANGUAGE:
        raise ValueError(
            f"Expected {SESSIONS_PER_LANGUAGE} sessions "
            f"for {language_code}"
        )

    if not (
        session_counts == SEGMENTS_PER_SESSION
    ).all():
        raise ValueError(
            f"Each session must contribute "
            f"{SEGMENTS_PER_SESSION} segments"
        )

    if result["normalized_text"].duplicated().any():
        raise ValueError(
            f"Duplicate transcript detected for {language_code}"
        )

    result["segment_timestamp"] = (
        result["segment_start"].map(format_timestamp)
        + " - "
        + result["segment_end"].map(format_timestamp)
    )

    result["verdict"] = pd.Series(
        "",
        index=result.index,
        dtype="string",
    )

    result["note"] = pd.Series(
        "",
        index=result.index,
        dtype="string",
    )

    return result[
        [
            "gamesession_id",
            "url",
            "segment_index",
            "segment_timestamp",
            "segment_text",
            "language_detected",
            "verdict",
            "note",
        ]
    ]


def show_segments(frame):
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
    ):
        display(frame)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
english_segments = get_language_segments("en")
show_segments(english_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141261607,https://www.youtube.com/watch?v=bvq10rcghiQ,811,02:40:12 - 02:40:15,Yeah. I'm big dead.,en,,
1,141261607,https://www.youtube.com/watch?v=bvq10rcghiQ,814,02:40:37 - 02:40:41,Where's kind of thick body at? way it off the map dude.,en,,
2,141261607,https://www.youtube.com/watch?v=bvq10rcghiQ,815,02:40:41 - 02:40:42,You gotta be kidding.,en,,
3,141261607,https://www.youtube.com/watch?v=bvq10rcghiQ,819,02:41:23 - 02:41:28,My solution is simple I call them over and give them a good talk.,en,,
4,141261607,https://www.youtube.com/watch?v=bvq10rcghiQ,820,02:41:30 - 02:41:36,"what's with that look? Relax, I ain't gonna do that to a partner?",en,,
5,141240944,https://www.youtube.com/watch?v=YtxE-T1fF24,450,00:54:18 - 00:54:24,"That's bullshit. a bruiser here, taking that shit. They're still not building.",en,,
6,141240944,https://www.youtube.com/watch?v=YtxE-T1fF24,451,00:54:25 - 00:54:28,"Yeah, they're going to the left. I see them.",en,,
7,141240944,https://www.youtube.com/watch?v=YtxE-T1fF24,452,00:54:29 - 00:54:34,"Now they're running the left. Oh, I them. hunting. just go the zone.",en,,
8,141240944,https://www.youtube.com/watch?v=YtxE-T1fF24,454,00:54:40 - 00:54:43,That's choking. There's being weirdos.,en,,
9,141240944,https://www.youtube.com/watch?v=YtxE-T1fF24,456,00:54:47 - 00:54:50,I I just heard this door open right in front us. No. No.,en,,


In [4]:
german_segments = get_language_segments("de")
show_segments(german_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141237699,https://www.youtube.com/watch?v=KUVIj4bkpaE,154,00:24:18 - 00:24:25,Okay. der kann man kann das nicht beheben,de,,
1,141237699,https://www.youtube.com/watch?v=KUVIj4bkpaE,160,00:26:10 - 00:26:17,unterhalten das war ein blog auf der gesamten med und jetzt reicht's,de,,
2,141237699,https://www.youtube.com/watch?v=KUVIj4bkpaE,162,00:26:33 - 00:26:39,Hallo? Gehst du jetzt los oder musst du jetzt so unterhalten?,de,,
3,141237699,https://www.youtube.com/watch?v=KUVIj4bkpaE,166,00:26:54 - 00:26:57,"Ich bin gleich bei Crazy Dudes erst. Nein, wir warten nicht...hä?",de,,
4,141237699,https://www.youtube.com/watch?v=KUVIj4bkpaE,167,00:26:58 - 00:27:02,Ich warte jetzt nicht. Es hat damit doch nichts...,de,,
5,141247704,https://www.youtube.com/watch?v=LeEZMbLafl4,140,00:25:32 - 00:25:36,"So, genug davon. Wir sollten wirklich nach Hause.",de,,
6,141247704,https://www.youtube.com/watch?v=LeEZMbLafl4,141,00:25:36 - 00:25:43,"Gehen wir. Warte. Was? Ezio, lass Christina schlafen.",de,,
7,141247704,https://www.youtube.com/watch?v=LeEZMbLafl4,142,00:25:43 - 00:25:46,Dafür bleibt schon genug Zeit. Nachher.,de,,
8,141247704,https://www.youtube.com/watch?v=LeEZMbLafl4,143,00:26:24 - 00:26:29,Cazzo. Vieri. Ich verstecke mich besser. Sucht weiter!,de,,
9,141247704,https://www.youtube.com/watch?v=LeEZMbLafl4,144,00:26:30 - 00:26:31,Er kann nicht weit gekommen sein.,de,,


In [5]:
french_segments = get_language_segments("fr")
show_segments(french_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141263069,https://www.youtube.com/watch?v=6Vfg1hHM4ao,129,00:25:32 - 00:25:37,J'avais envie au maximum !,fr,,
1,141263069,https://www.youtube.com/watch?v=6Vfg1hHM4ao,130,00:25:37 - 00:25:44,Le mec il m'a... Le mec...,fr,,
2,141263069,https://www.youtube.com/watch?v=6Vfg1hHM4ao,131,00:25:44 - 00:25:49,Oh non et j'ai perdu le fusil d'assaut ! Putain !,fr,,
3,141263069,https://www.youtube.com/watch?v=6Vfg1hHM4ao,132,00:25:49 - 00:25:53,Bon bah je pense que c le moment de galérer là !,fr,,
4,141263069,https://www.youtube.com/watch?v=6Vfg1hHM4ao,134,00:26:00 - 00:26:03,c'était pas un mystère hein... merde...,fr,,
5,141240286,https://www.youtube.com/watch?v=LmEEXo-Vbzs,79,00:19:34 - 00:19:42,"Oh bordel, qu'est que je suis joué !",fr,,
6,141240286,https://www.youtube.com/watch?v=LmEEXo-Vbzs,80,00:19:42 - 00:19:46,Alléluia ! J'ai trouvé le chemin !,fr,,
7,141240286,https://www.youtube.com/watch?v=LmEEXo-Vbzs,82,00:19:54 - 00:19:59,Mais qu-ce que je suis pas adoué des fois ? Quoi ?,fr,,
8,141240286,https://www.youtube.com/watch?v=LmEEXo-Vbzs,84,00:20:12 - 00:20:17,Putain j'ai perdu combien de temps pour trouver le chemin ? 20 minutes ?,fr,,
9,141240286,https://www.youtube.com/watch?v=LmEEXo-Vbzs,89,00:21:38 - 00:21:44,oh Tu l'as dit Quoi,fr,,


In [6]:
portuguese_segments = get_language_segments("pt")
show_segments(portuguese_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141262388,https://www.youtube.com/watch?v=MDLhox_0yKk,541,02:05:49 - 02:05:53,"ó garoto, marca eu nesse poxa que me republicar quiser",pt,,
1,141262388,https://www.youtube.com/watch?v=MDLhox_0yKk,545,02:07:21 - 02:07:26,viado sincronia de outro mundo O cara aqui Zé,pt,,
2,141262388,https://www.youtube.com/watch?v=MDLhox_0yKk,546,02:07:32 - 02:07:35,Rodou nele viado Onde que ele ta Zé?,pt,,
3,141262388,https://www.youtube.com/watch?v=MDLhox_0yKk,547,02:07:35 - 02:07:41,"Ta em baixo, ta em baixo É ruim?",pt,,
4,141262388,https://www.youtube.com/watch?v=MDLhox_0yKk,548,02:07:41 - 02:07:49,É ruim bosta vi haha O cara ai meu zé Matou eu viado,pt,,
5,141261605,https://www.youtube.com/watch?v=DPM1h5wViaw,475,01:54:13 - 01:54:20,vamos estamos chegando senhor a Grupo?,pt,,
6,141261605,https://www.youtube.com/watch?v=DPM1h5wViaw,476,01:54:20 - 01:54:24,"Sim senhor! Sim senhor! improvisar, Tenente?",pt,,
7,141261605,https://www.youtube.com/watch?v=DPM1h5wViaw,477,01:54:24 - 01:54:28,Afirmativo. O pelotão Epo está em curralado. gente vai conseguir tirar eles dessa.,pt,,
8,141261605,https://www.youtube.com/watch?v=DPM1h5wViaw,478,01:54:30 - 01:54:35,"Você está bem? Vou ficar bem. Bora, pega a direção.",pt,,
9,141261605,https://www.youtube.com/watch?v=DPM1h5wViaw,479,01:54:35 - 01:54:38,"Vamos então! Ah, vocês vão deixar eu dirigir?",pt,,


In [7]:
spanish_segments = get_language_segments("es")
show_segments(spanish_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141261474,https://www.youtube.com/watch?v=KtAWOITWjqk,691,02:58:47 - 02:58:51,"Néstor seis, arroba, J, Brito, ¿qué?",es,,
1,141261474,https://www.youtube.com/watch?v=KtAWOITWjqk,693,02:58:58 - 02:59:02,hace ese tipo con el que estás? ¿Y qué hace?,es,,
2,141261474,https://www.youtube.com/watch?v=KtAWOITWjqk,699,03:00:20 - 03:00:24,crujete 5 online says del dale ntp,es,,
3,141261474,https://www.youtube.com/watch?v=KtAWOITWjqk,701,03:00:46 - 03:00:48,como que se escribe la rayita abajo?,es,,
4,141261474,https://www.youtube.com/watch?v=KtAWOITWjqk,702,03:00:58 - 03:01:06,me olvido como que uno hace la rayita abajo,es,,
5,141261469,https://www.youtube.com/watch?v=N4GZLs8diH0,415,01:30:07 - 01:30:11,mataste al conejo? te da uno nomas?,es,,
6,141261469,https://www.youtube.com/watch?v=N4GZLs8diH0,419,01:31:12 - 01:31:16,vamos arriba que hay gente aca arriba,es,,
7,141261469,https://www.youtube.com/watch?v=N4GZLs8diH0,421,01:31:36 - 01:31:44,me espantio otra vez el humo o no lo vi? esta ahi adentro,es,,
8,141261469,https://www.youtube.com/watch?v=N4GZLs8diH0,422,01:31:54 - 01:31:58,"Bueno, bueno, bueno, bueno, bueno.",es,,
9,141261469,https://www.youtube.com/watch?v=N4GZLs8diH0,423,01:31:58 - 01:32:00,"Dame un toque, dame un toque, dame un toque.",es,,


In [8]:
russian_segments = get_language_segments("ru")
show_segments(russian_segments)


ValueError: Not enough qualifying YouTube sessions for ru: 2/5